## **Problem Statement**

### Business Context

The prices of the stocks of companies listed under a global exchange are influenced by a variety of factors, with the company's financial performance, innovations and collaborations, and market sentiment being factors that play a significant role. News and media reports can rapidly affect investor perceptions and, consequently, stock prices in the highly competitive financial industry. With the sheer volume of news and opinions from a wide variety of sources, investors and financial analysts often struggle to stay updated and accurately interpret its impact on the market. As a result, investment firms need sophisticated tools to analyze market sentiment and integrate this information into their investment strategies.

### Problem Definition

With an ever-rising number of news articles and opinions, an investment startup aims to leverage artificial intelligence to address the challenge of interpreting stock-related news and its impact on stock prices. They have collected historical daily news for a specific company listed under NASDAQ, along with data on its daily stock price and trade volumes.

As a member of the Data Science and AI team in the startup, you have been tasked with developing an AI-driven sentiment analysis system that will automatically process and analyze news articles to gauge market sentiment, and summarizing the news at a weekly level to enhance the accuracy of their stock price predictions and optimize investment strategies. This will empower their financial analysts with actionable insights, leading to more informed investment decisions and improved client outcomes.

### Data Dictionary

* `Date` : The date the news was released
* `News` : The content of news articles that could potentially affect the company's stock price
* `Open` : The stock price (in \$) at the beginning of the day
* `High` : The highest stock price (in \$) reached during the day
* `Low` :  The lowest stock price (in \$) reached during the day
* `Close` : The adjusted stock price (in \$) at the end of the day
* `Volume` : The number of shares traded during the day
* `Label` : The sentiment polarity of the news content
    * 1: positive
    * 0: neutral
    * -1: negative

## **Installing and Importing the necessary libraries**

In [ ]:
# installing the sentence-transformers and gensim libraries for word embeddings
!pip install numpy==1.26.4 \
             scikit-learn==1.6.1 \
             scipy==1.13.1 \
             gensim==4.3.3 \
             sentence-transformers==3.4.1 \
             pandas==2.2.2

In [ ]:
# to read and manipulate the data
import numpy as np
import pandas as pd
pd.set_option('max_colwidth', None)    # setting column to the maximum column width as per the data

# to visualise data
import matplotlib.pyplot as plt
import seaborn as sns

# To import Word2Vec
from gensim.models import Word2Vec

# Deep Learning library
import torch

# to load transformer models
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration, pipeline

# To split data into train and test sets
from sklearn.model_selection import train_test_split

# To build a Random Forest model
from sklearn.ensemble import RandomForestClassifier

# To compute metrics to evaluate the model
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

# to ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

# Import TensorFlow and Keras for deep learning model building.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

## **Loading the dataset**

In [ ]:
# Mount Google Drive to the /content/drive directory to access the files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# loading data into a pandas dataframe
News = pd.read_csv("/content/drive/MyDrive/Gen_Ai/stock_news.csv")

In [ ]:
# creating a copy of the data
data = News.copy()

## **Data Overview**

###**Checking the first 5 rows**

In [ ]:
data.head(5)

###**Checking the last 5 rows**

In [ ]:
data.tail(5)

###**Checking the shape of the data**

In [ ]:
data.shape

###**Checking for missing values**

In [ ]:
News.isnull().sum()

### **Checking for duplicate values**

In [ ]:
# checking for duplicate values
data.duplicated().sum()

## **Exploratory Data Analysis**

In [ ]:
# To display the structure of data
data.info()

In [ ]:
#converting object column 'Date' to datetime
data['Date'] = pd.to_datetime(data['Date'])

In [ ]:
# To ensure the data type of Date
data.dtypes

In [ ]:
# summary statistics for each numeric column
data.describe()

### **Univariate Analysis**

* Distribution of individual variables
* Compute and check the distribution of the length of news content
* Monthwise Analysis

### **Distribution of Sentiment polarity and all Numeric Variables**

In [ ]:
# Distribution of sentiment polarity
sns.countplot(x='Label', data=data)
plt.title("Distribution of Sentiment Polarity")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
# Plot Distribution of all Numeric variables
numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']

data[numeric_columns].hist(figsize=(12,8), bins=20)
plt.suptitle("Distribution of Numeric Variables")
plt.show()


### **Distribution of the length of news content**

In [ ]:
# Compute length of news content (in words)

data['News_Length'] = data['News'].apply(lambda x: len(str(x).split()))

# Summary statistics of news length
data['News_Length'].describe()



In [ ]:
# Plot distribution
data['News_Length'] = data['News'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10,6))
sns.histplot(data['News_Length'], bins=30, kde=True, color='skyblue')
plt.title("Distribution of News Content Length")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


### **Monthwise Analysis**

In [ ]:
#Create a YearMonth column
data['YearMonth'] = data['Date'].dt.to_period('M').astype(str)

# Group by YearMonth and calculate averages
monthwise_avg = (
    data.groupby('YearMonth')[['High', 'Low', 'Open', 'Close','Volume']]
      .mean()
      .reset_index()
)

print(monthwise_avg)

In [ ]:
# plot the monthwise averages of all price variables for better visualization
price_columns = ['Open', 'High', 'Low', 'Close']

monthwise_avg = data.groupby('YearMonth')[price_columns].mean()

monthwise_avg.plot(figsize=(10,6), marker='o')
plt.title("Monthwise Average of Stock Price Variables")
plt.xlabel("Year-Month")
plt.ylabel("Average Price")
plt.grid(True)
plt.show()


* Price Trend (High, Low, Open, Close) shows that there is a steady increase in all price measures from January to April 2019.

In [ ]:
#plot the trend of monthwise averages in Volume
volume_avg = data.groupby('YearMonth')['Volume'].mean()

plt.figure(figsize=(10,5))
volume_avg.plot(marker='o', color='green')
plt.title("Monthwise Average Volume Trend")
plt.xlabel("Year-Month")
plt.ylabel("Average Volume")
plt.grid(True)
plt.show()

In [ ]:
# Bar plot of the monthwise averages in Volume
volume_avg = data.groupby('YearMonth')['Volume'].mean()

plt.figure(figsize=(10,5))
volume_avg.plot(kind='bar', color='skyblue')
plt.title("Monthwise Average Trading Volume")
plt.xlabel("Year-Month")
plt.ylabel("Average Volume")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.show()


In [ ]:
# Boxplot to identify outliers
numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']

plt.figure(figsize=(12,6))
data[numeric_columns].boxplot()
plt.title("Boxplot of Numeric Variables")
plt.ylabel("Values")
plt.xticks(rotation=45)
plt.show()

### **Bivariate Analysis**

* Correlation
* Sentiment Polarity vs Price
* Date vs Price
* Distribution of News Length Across Sentiment Categories



### **Correlation**

In [ ]:
# Correlation matrix to check if variables move together or have correlations
numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']

correlation_matrix = data[numeric_columns].corr()

plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix of Numeric Variables")
plt.show()

### **Sentiment Polarity vs Price**

In [ ]:
#Sentiment Polarity with all numeric variables
numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
sentiment_analysis = data.groupby('Label')[numeric_columns].mean()

plt.figure(figsize=(10,6))
sns.heatmap(sentiment_analysis, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title("Sentiment Polarity vs Numeric Variables")
plt.xlabel("Numeric Variables")
plt.ylabel("Sentiment Polarity")
plt.show()


### **Time Series Analysis (Date vs Price)**

In [ ]:
#Time Series Analysis (Date vs Price columns)
price_columns = ['Open', 'High', 'Low', 'Close']

plt.figure(figsize=(12,6))

for column in price_columns:
    plt.plot(data['Date'], data[column], label=column)

plt.title("Time Series Analysis of Stock Prices")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.legend()
plt.grid(True)
plt.show()


### **Distribution of News Length Across Sentiment Categories**

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='Label', y='News_Length', data=data)

plt.title("Distribution of News Length Across Sentiment Categories")
plt.xlabel("Sentiment Polarity")
plt.ylabel("News Length (Number of Words)")
plt.show()

## **Data Preprocessing**

* Split the dataset into training and test sets using an **80:20** ratio with **train_test_split** (random_state=42) to ensure reproducibility.

* The feature matrix  **X** contains text embeddings generated from the embedding stage.

* The dimensionality of **X** varies depending on the embedding model used.

* For each embedding type, separate **X_train and X_test splits** were created, paired with the target **y_train, y_test**.

In [ ]:
# Storing target variable
y = data['Label']

## **Word Embeddings**


#### **Word2Vec**

In [ ]:
# Creating a list of all words in our data
sentences = [text.split() for text in data['News']]

In [ ]:
# Import Word2Vec
from gensim.models import Word2Vec
# Train the Word2Vec model
word2vec_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0
)

In [ ]:
# Display information about the trained Word2Vec model
print("Vocabulary Size:", len(word2vec_model.wv))
print("Vector Dimension:", word2vec_model.vector_size)

# Display a sample word vector
print("\nVector for the word 'stock':")
print(word2vec_model.wv['stock'])

In [ ]:

# Retrieving word vectors for all the words present in the model's vocabulary
word_vectors = word2vec_model.wv

# Creating a dictionary of words and their corresponding vectors
word_vector_dict = {word: word_vectors[word] for word in word_vectors.index_to_key}

In [ ]:
def average_vectorizer_Word2Vec(doc):
    # Initializing a feature vector for the sentence
    feature_vector = np.zeros((vec_size,), dtype="float64")

    # Creating a list of words in the sentence that are present in the model vocabulary
    words_in_vocab = [word for word in doc.split() if word in words]

    # adding the vector representations of the words
    for word in words_in_vocab:
        feature_vector += np.array(word_vector_dict[word])

    # Dividing by the number of words to get the average vector
    if len(words_in_vocab) != 0:
        feature_vector /= len(words_in_vocab)

    return feature_vector

In [ ]:
# creating a dataframe of the vectorized documents
x = pd.Daword_vector_dict={word : word_vectors[word] for word in word_vectors.index_to_key}

## **Sentence Transformer**

####**Generating Embeddings with Sentence Tranformers using the BAAI/bge-base-en-v1.5 text embedding**

In [ ]:
model = SentenceTransformer('')

##### Encoding the dataset

In [ ]:
embedding_matrix = model.encode(
    data['News'].tolist(),
    show_progress_bar=True
)

In [ ]:
# printing the shape of the embedding matrix
embedding_matrix.shape

## **Sentiment Analysis**

### **Model Evaluation Criterion**


- Plot a **confusion matrix** (`plot_confusion_matrix`)
- Generate key **classification metrics** like accuracy, recall, precision, and F1-score (`model_performance_classification_sklearn`)



##### **Utility Functions**

In [ ]:
def plot_confusion_matrix(actual, predicted):
    """
    Plot a confusion matrix to visualize the performance of a classification model.

    Parameters:
    actual (array-like): The true labels.
    predicted (array-like): The predicted labels from the model.

    Returns:
    None: Displays the confusion matrix plot.
    """

    # Compute the confusion matrix.
    cm = confusion_matrix(actual, predicted)

    # Create a new figure with a specified size
    plt.figure(figsize=(5, 4))

    # Define the labels for the confusion matrix dynamically from the data
    label_list = sorted(list(np.unique(np.concatenate((actual, predicted)))))

    # Plot the confusion matrix using a heatmap with annotations
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', xticklabels=label_list, yticklabels=label_list)

    # Label for the y-axis
    plt.ylabel('Actual')

    # Label for the x-axis
    plt.xlabel('Predicted')

    # Title of the plot
    plt.title('Confusion Matrix')

    # Display the plot
    plt.show()

In [ ]:
def model_performance_classification_sklearn(actual, predicted):
    """
    Compute various performance metrics for a classification model using sklearn.

    Parameters:
    model (sklearn classifier): The classification model to evaluate.
    predictors (array-like): The independent variables used for predictions.
    target (array-like): The true labels for the dependent variable.

    Returns:
    pandas.DataFrame: A DataFrame containing the computed metrics (Accuracy, Recall, Precision, F1-score).
    """

    # Compute Accuracy
    acc = accuracy_score(actual,predicted)
    # Compute Recall
    recall = recall_score(actual,predicted,average='weighted')
    # Compute Precision
    precision = precision_score(actual,predicted,average='weighted')
    # Compute F1-score
    f1 = f1_score(actual,predicted,average='weighted')

    # Create a DataFrame to store the computed metrics
    df_perf = pd.DataFrame(
        {
            "Accuracy": [acc],
            "Recall": [recall],
            "Precision": [precision],
            "F1": [f1],
        }
    )
    # Return the DataFrame with the metrics
    return df_perf

## **Build Random Forest Models using different text embeddings**

### **Word2Vec-Random Forest Model**

In [ ]:
# Storing independent variable
vec_size = word2vec_model.vector_size
words = word_vector_dict.keys()
X = pd.DataFrame([average_vectorizer_Word2Vec(doc) for doc in data['News']])

In [ ]:
# Split data into training and testing set.
X_train, X_test, y_train, y_test = train_test_split(X ,y, test_size = 0.2, random_state = 42)

In [ ]:
# Building the model
rf_w2v_model = RandomForestClassifier(n_estimators=100, random_state=42)
# Fitting on train data
rf_w2v_model.fit(X_train, y_train)

In [ ]:
# Predicting on train data
y_pred_train = rf_w2v_model.predict(X_train)

# Predicting on test data
y_pred_test = rf_w2v_model.predict(X_test)

In [ ]:
plot_confusion_matrix(y_train, y_pred_train)

In [ ]:
plot_confusion_matrix(y_test, y_pred_test)

In [ ]:
train_Word2Vec_RF = model_performance_classification_sklearn(y_train,y_pred_train)
print("Training set performance metrics:")
print(train_Word2Vec_RF.to_string(index=False))
test_Word2Vec_RF = model_performance_classification_sklearn(y_test,y_pred_test)
print("Test set performance metrics:")
print(test_Word2Vec_RF.to_string(index=False))

###**Sentence Transformer-Random Forest Model**

In [ ]:
# Storing independent variable
X = embedding_matrix


In [ ]:
# Split data into training and testing set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Building the model
rf_transformer_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fitting on train data
rf_transformer_model.fit(X_train, y_train)

In [ ]:
# Predicting on train data

y_pred_train = rf_transformer_model.predict(X_train)
# Predicting on test data
y_pred_test = rf_transformer_model.predict(X_test)

In [ ]:
plot_confusion_matrix(y_train, y_pred_train)

In [ ]:
plot_confusion_matrix(y_test, y_pred_test)

In [ ]:
train_transformer_BAAI_RF = model_performance_classification_sklearn(y_train,y_pred_train)
print("Training set performance metrics:")
print(train_transformer_BAAI_RF.to_string(index=False))
test_transformer_BAAI_RF = model_performance_classification_sklearn(y_test,y_pred_test)
print("Test set performance metrics:")
print(test_transformer_BAAI_RF.to_string(index=False))


### **Building Neural Network Models using different text embeddings**

### **Word2Vec-Neural Network Model**

In [ ]:
# Storing independent variable
vec_size = word2vec_model.vector_size
words = word_vector_dict.keys()
X = pd.DataFrame([average_vectorizer_Word2Vec(doc) for doc in data['News']])

In [ ]:
# Split data into training and testing set.
X_train, X_test, y_train, y_test = train_test_split(X ,y, test_size = 0.2, random_state = 42)

In [ ]:
label_map = {-1: 0, 0: 1, 1: 2}
y_train_mapped = np.array([label_map[y] for y in y_train])
y_test_mapped = np.array([label_map[y] for y in y_test])

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')  # 3 output neurons for 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train_mapped,
    validation_data=(X_test, y_test_mapped),
    epochs=20,
    batch_size=32
)

In [ ]:
# Predict class probabilities on training data
y_train_pred_probs = model.predict(X_train)

# Convert probabilities to class labels
y_train_preds = np.argmax(y_train_pred_probs, axis=1)

In [ ]:
# Predict class probabilities on test data
y_test_pred_probs = model.predict(X_test)

# Convert probabilities to class labels
y_test_preds = np.argmax(y_test_pred_probs, axis=1)

In [ ]:
plot_confusion_matrix(y_train_mapped, y_train_preds)

In [ ]:
plot_confusion_matrix(y_test_mapped, y_test_preds)

In [ ]:
train_Word2Vec_NN = model_performance_classification_sklearn(y_train_mapped, y_train_preds)
print("Training set performance metrics:")
print(train_Word2Vec_NN .to_string(index=False))
test_Word2Vec_NN  = model_performance_classification_sklearn(y_test_mapped, y_test_preds)
print("Test set performance metrics:")
print(test_Word2Vec_NN .to_string(index=False))



### **Generating Embeddings with Sentence Tranformers using the BAAI/bge-base-en-v1.5 embedding -Neural Network model**

In [ ]:
# Split data into training and testing set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
label_map = {-1: 0, 0: 1, 1: 2}
y_train_mapped = np.array([label_map[y] for y in y_train])
y_test_mapped = np.array([label_map[y] for y in y_test])

In [ ]:
import gc

# Clear previous sessions
tf.keras.backend.clear_session()
gc.collect()

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')  # 3 output neurons for 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train_mapped,
    validation_data=(X_test, y_test_mapped),
    epochs=20,
    batch_size=32
)

In [ ]:
# Predict class probabilities on training data
y_train_pred_probs = model.predict(X_train)

# Convert probabilities to class labels
y_train_preds = np.argmax(y_train_pred_probs, axis=1)

In [ ]:
# Predict class probabilities on test data
y_test_pred_probs = model.predict(X_test)

# Convert probabilities to class labels
y_test_preds = np.argmax(y_test_pred_probs, axis=1)

In [ ]:
plot_confusion_matrix(y_train_mapped, y_train_preds)

In [ ]:
plot_confusion_matrix(y_test_mapped, y_test_preds)

In [ ]:
train_transformer_BAAI_NN = model_performance_classification_sklearn(y_train_mapped, y_train_preds)
print("Training set performance metrics:")
print(train_transformer_BAAI_NN .to_string(index=False))
test_transformer_BAAI_NN  = model_performance_classification_sklearn(y_test_mapped, y_test_preds)
print("Test set performance metrics:")
print(test_transformer_BAAI_NN .to_string(index=False))


### **Model Performance Summary and Final Model Selection**

In [ ]:
# Concatenate the training performance metrics from different models into a single DataFrame
models_train_comp_df = pd.concat(
    [
        train_Word2Vec_RF.T,  # Random Forest using Word2Vec embeddings
        train_Word2Vec_NN.T,  # Neural Network using Word2Vec embeddings
        train_transformer_BAAI_RF.T,  # Random Forest using Sentence-Transformer-BAAI/bge-base-en-v1.5 embeddings
        train_transformer_BAAI_NN.T, # Neural Network using Sentence-Transformer-BAAI/bge-base-en-v1.5 embeddings
        train_transformer_MiniLM_RF.T, # Random Forest using Sentence-Transformer-all-MiniLM-L6-v2 embeddings
        train_transformer_MiniLM_NN.T   # Neural Network using Sentence-Transformer-all-MiniLM-L6-v2 embeddings
    ],
    axis=1  # Concatenate along columns (i.e., each model's metrics form one column)
)

# Assigning column names for each model
models_train_comp_df.columns = [
    "Word2Vec (Random Forest)",
    "Word2Vec (Neural Network)",
    "Sentence Transformer-BAAI/bge-base-en-v1.5 (Random Forest)",
    "Sentence Transformer-BAAI/bge-base-en-v1.5 (Neural Network)",
    "Sentence Transformer-all-MiniLM-L6-v2 (Random Forest)",
    "Sentence Transformer-all-MiniLM-L6-v2 (Neural Network)"
]

# Print the training performance comparison table
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# Concatenate the test performance metrics from different models into a single DataFrame
models_test_comp_df = pd.concat(
    [
        test_Word2Vec_RF.T,  # Random Forest using Word2Vec embeddings
        test_Word2Vec_NN.T,  # Neural Network using Word2Vec embeddings
        test_transformer_BAAI_RF.T,  # Random Forest using Sentence-Transformer-BAAI/bge-base-en-v1.5 embeddings
        test_transformer_BAAI_NN.T, # Neural Network using Sentence-Transformer-BAAI/bge-base-en-v1.5 embeddings
        test_transformer_MiniLM_RF.T, # Random Forest using Sentence-Transformer-all-MiniLM-L6-v2 embeddings
        test_transformer_MiniLM_NN.T   # Neural Network using Sentence-Transformer-all-MiniLM-L6-v2 embeddings
    ],
    axis=1  # Concatenate along columns (i.e., each model's metrics form one column)
)

# Assigning column names for each model
models_test_comp_df.columns = [
    "Word2Vec (Random Forest)",
    "Word2Vec (Neural Network)",
    "Sentence Transformer-BAAI/bge-base-en-v1.5 (Random Forest)",
    "Sentence Transformer-BAAI/bge-base-en-v1.5 (Neural Network)",
    "Sentence Transformer-all-MiniLM-L6-v2 (Random Forest)",
    "Sentence Transformer-all-MiniLM-L6-v2 (Neural Network)"
]

# Print the training performance comparison table
print("Test performance comparison:")
models_test_comp_df

## **Conclusions and Recommendations**

#### **Conclusions**

**Word2Vec**

**Sentence Transformer embeddings**

**Neural Networks**

**Random Forest**



#### **Recommendations**